In [18]:
import os
import sys

# Tự động cấu hình Java và Hadoop (winutils) cho Spark trên Windows
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
os.environ["HADOOP_HOME"] = r"D:\NYC_Taxi_Prj\hadoop"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.path.join(os.environ["HADOOP_HOME"], "bin")

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("nyc-taxi")
    .getOrCreate()
)


In [24]:
df = spark.read.parquet(
    "data/raw/yellow_tripdata_2026-01.parquet"
)

In [25]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [26]:
df.show(4)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [ ]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 3724889
Columns: 20


In [ ]:
df.describe().show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+------------------+--------------------+-------------------+------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|     DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee|cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+------

In [ ]:
df2 = (
    df

    .withColumn(
        "trip_duration_min",

        (
            unix_timestamp(
                "tpep_dropoff_datetime"
            )

            -

            unix_timestamp(
                "tpep_pickup_datetime"
            )

        )/60
    )# Lấy thời gian trả khách - thời gian nhận đón khách

    .withColumn(
        "tip_ratio",

        col("tip_amount")
        /

        col("fare_amount")
    )

    .withColumn(
        "pickup_date",

        to_date(
            "tpep_pickup_datetime"
        )
    )
)

In [ ]:
df2.select(
"trip_duration_min",
"tip_ratio",
"pickup_date"
).show(10)

+------------------+-------------------+-----------+
| trip_duration_min|          tip_ratio|pickup_date|
+------------------+-------------------+-----------+
|              5.55| 0.5083333333333333| 2026-01-01|
| 5.716666666666667|                0.0| 2026-01-01|
| 8.883333333333333|0.23364485981308414| 2026-01-01|
|              42.8| 0.2870801033591731| 2026-01-01|
|              13.5| 0.2851851851851852| 2026-01-01|
|              13.6| 0.3514084507042254| 2026-01-01|
|10.633333333333333|                0.0| 2026-01-01|
|24.616666666666667|               0.25| 2026-01-01|
|37.733333333333334|0.23083109919571046| 2026-01-01|
| 9.583333333333334| 0.2205607476635514| 2026-01-01|
+------------------+-------------------+-----------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import *

df_different= df.select(
(
col("fare_amount")
+
col("extra")
+
col("mta_tax")
+
col("tip_amount")
+
col("tolls_amount")
+
col("improvement_surcharge")
+
col("congestion_surcharge")
+
col("Airport_fee")
+
col("cbd_congestion_fee")

-

col("total_amount")

).alias("diff")

)

In [ ]:
from pyspark.sql.functions import *

check_df = df.withColumn(

"diff",

col("fare_amount")
+
col("extra")
+
col("mta_tax")
+
col("tip_amount")
+
col("tolls_amount")
+
col("improvement_surcharge")
+
col("congestion_surcharge")
+
col("Airport_fee")
+
col("cbd_congestion_fee")

-

col("total_amount")
)

In [ ]:
check_df.select (

"fare_amount"
,
"extra"
,
"mta_tax"
,
"tip_amount"
,
"tolls_amount"
,
"improvement_surcharge"
,
"congestion_surcharge"
,
"Airport_fee"
,
"cbd_congestion_fee"
,"diff"
).where(
    col("diff").isNotNull()
    ).orderBy(
    col("diff").asc()
).show(10)

+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------------+------------------+
|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|congestion_surcharge|Airport_fee|cbd_congestion_fee|              diff|
+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------------+------------------+
|       70.0|  0.0|    0.5|     17.44|        7.46|                  1.0|                 2.5|        0.0|              0.75|-5.000000000000014|
|       70.0|  0.0|    0.5|     17.04|        6.94|                  1.0|                 0.0|       1.75|               0.0|-5.000000000000014|
|       70.0|  0.0|    0.5|     26.16|        7.46|                  1.0|                 2.5|        0.0|              0.75|-5.000000000000014|
|       70.0|  0.0|    0.5|     17.79|        7.46|                  1.0|                 2.5|       1.75|              0.75|-5.00